## RAG Application using Typesense

In [1]:
import typesense

In [ ]:
client = typesense.Client({
    "nodes": [{
        'host': 'oqt0vi3n7gz2k6esp-1.a2.typesense.net',
        'port': '443',
        'protocol': 'https'
    }],
    "api_key": 'vJGF0jyTkmZFLZUty6aAIu2boZCE8Itb',
    "connection_timeout_seconds": 2
})

book_schema = {
    'name':'books',
    'fields': [
        {'name': 'title', 'type': 'string'},
        {'name': 'authors', 'type': 'string[]', 'facet':True},
        {'name': 'publication_year', 'type': 'int32', 'facet':True},
        {'name': 'ratings_count', 'type': 'int32'},
        {'name': 'average_rating', 'type': 'float'}
    ],
    'default_sorting_field': 'ratings_count'
}

print(client.collections.create(book_schema))

{'created_at': 1775244779, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [4]:
with open('books.jsonl', 'r', encoding='utf-8') as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [8]:
search_parameters = {
    'q': 'harry potter',
    'query_by': 'title,authors',
    'filter_by': 'publication_year:<1998',
    'sort_by': 'ratings_count:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [9]:
search_parameters = {
    'q': 'experyment',
    'query_by': 'title',
    'facet_by': 'authors',
    'sort_by': 'ratings_count:desc'
}

client.collections['books'].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787300

In [14]:
#Langchain + Typesense + Groq LLM 
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [15]:
import os
groq_api_key = os.environ['GROQ_API_KEY']

In [16]:
loader = TextLoader('cricket.txt')
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

In [17]:
doc_search = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        'host': 'oqt0vi3n7gz2k6esp-1.a2.typesense.net',
        'port': '443',
        'protocol': 'https',
        'typesense_api_key': 'vJGF0jyTkmZFLZUty6aAIu2boZCE8Itb',
        'typesense_collection_name': 'langchain'
    },
)

In [19]:
query = "What is cricket?"
found_docs = doc_search.similarity_search(query)
found_docs[0].page_content

'Cricket is one of the most popular sports in the world. It is played and loved by millions of people across many countries. The game is especially famous in nations like India, Australia, England, Pakistan, and New Zealand.\n\nCricket is played between two teams of eleven players each. It requires skill, teamwork, patience, and strategy. Fans enjoy watching thrilling matches and supporting their favorite teams and players.\n\nHistory of Cricket\n\nCricket originated in England during the 16th century. Over time, the sport spread to many countries through British influence. Today, cricket is governed internationally by the International Cricket Council.\n\nModern cricket is played in three main formats: Test matches, One Day Internationals (ODIs), and Twenty20 (T20). Each format has its own style and excitement.\n\nHow Cricket is Played'

In [20]:
#Retriever
retriever = doc_search.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x34a874b10>, search_kwargs={})

In [21]:
query = "Who wom most world cups?"
retriever.invoke(query)[0]

Document(metadata={'source': 'cricket.txt'}, page_content='2. India\nIndia national cricket team is one of the most popular teams in the world. India has won the ICC Cricket World Cup 2 times (1983 and 2011). The team has produced many legendary players and has a huge fan base.\n\n3. West Indies\nWest Indies cricket team dominated world cricket in the 1970s and 1980s. They won the first two ICC Cricket World Cups in 1975 and 1979.\n\n4. Pakistan\nPakistan national cricket team won the ICC Cricket World Cup in 1992. Pakistan is known for producing some of the best fast bowlers in cricket history.\n\n5. England\nEngland cricket team, the country where cricket started, won their first ICC Cricket World Cup in 2019 after a dramatic final match.\n\n6. Sri Lanka\nSri Lanka national cricket team won the ICC Cricket World Cup in 1996 and became one of the strongest teams in Asia.\n\nImportance of Cricket')